In [ ]:
import os

# 创建保存文件的目录（可选）
os.makedirs('../pythorch/data', exist_ok=True)

# 指定文件路径
file_path = os.path.join('../pythorch/data', 'house_tiny.csv')
# 写入CSV内容
with open(file_path, 'w') as f:
    f.write('NumRooms,Alley,Price\n')
    f.write('NA,Pave,127500\n')
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

print(f'CSV 文件已创建：{file_path}')

CSV 文件已创建：data/house_tiny.csv


In [2]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import os

# 定义一个自定义的 Dataset 类，用于处理CSV文件
class CsvDataset(Dataset):
    """
    一个用于处理CSV数据的自定义Dataset类。
    """
    def __init__(self, file_path):
        """
        初始化数据集。
        Args:
            file_path (str): CSV文件的路径。
        """
        # 检查文件是否存在
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件未找到: {file_path}")

        # 使用pandas读取CSV文件
        self.data_frame = pd.read_csv(file_path)

        # 先将分类数据（如'Alley'列）转换为数值
        # 注意：这里只是一个简单的示例，实际应用中需要更复杂的编码
        self.data_frame = pd.get_dummies(self.data_frame, columns=['Alley'])
        # 将所有数据尝试转换为数值类型，无法转换的变为NaN
        self.data_frame = self.data_frame.apply(pd.to_numeric, errors='coerce')
        # 用0填充所有NA值
        self.data_frame.fillna(0, inplace=True)

        # 将数据和标签分开
        # 假设最后一列是标签（Price）
        self.features = self.data_frame.iloc[:, :-1].values
        self.labels = self.data_frame.iloc[:, -1].values

        # 将数据转换为PyTorch张量
        self.features_tensor = torch.tensor(self.features, dtype=torch.float32)
        self.labels_tensor = torch.tensor(self.labels, dtype=torch.float32)


    def __len__(self):
        """
        返回数据集的样本总数。
        """
        return len(self.data_frame)

    def __getitem__(self, index):
        """
        根据索引返回一个数据样本。
        Args:
            index (int): 样本的索引。
        Returns:
            tuple: (features, label) 的元组。
        """
        features = self.features_tensor[index]
        label = self.labels_tensor[index]
        return features, label

# --- 如何使用这个自定义的Dataset类 ---

# 假设你已经创建了 'house_tiny.csv' 文件
# 创建文件路径
# os.makedirs(os.path.join('..', 'data'), exist_ok=True)
# data_file = os.path.join('..', 'data', 'house_tiny.csv')
# with open(data_file, 'w') as f:
#     f.write('NumRooms,Alley,Price\n')
#     f.write('NA,Pave,127500\n')
#     f.write('2,NA,106000\n')
#     f.write('4,NA,178100\n')
#     f.write('NA,NA,140000\n')

file_path = os.path.join('../pythorch/data', 'house_tiny.csv')

# 创建一个数据集实例
try:
    dataset = CsvDataset(file_path)
    print(f"数据集总共有 {len(dataset)} 个样本。")

    # 通过索引访问数据集中的样本
    first_sample = dataset[0]
    print(f"\n第一个样本的特征: {first_sample[0]}")
    print(f"第一个样本的标签: {first_sample[1]}")

    second_sample = dataset[1]
    print(f"\n第二个样本的特征: {second_sample[0]}")
    print(f"第二个样本的标签: {second_sample[1]}")

except FileNotFoundError as e:
    print(e)

数据集总共有 4 个样本。

第一个样本的特征: tensor([     0., 127500.])
第一个样本的标签: 1.0

第二个样本的特征: tensor([2.0000e+00, 1.0600e+05])
第二个样本的标签: 0.0
